In [ ]:
# Instalacion de dependencias
import importlib.util
import subprocess
import sys

packages = {
    'cv2': 'opencv-python',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'tensorflow': 'tensorflow',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
}

missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', *missing],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )


In [ ]:
from pathlib import Path
import json
import re
import shutil
import unicodedata
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

ZIP_PATH = Path('/content/Videos.zip') if Path('/content').exists() else Path('Videos.zip')

if not ZIP_PATH.exists() and Path('/content').exists():
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        uploaded_name = next(iter(uploaded))
        Path(uploaded_name).rename(ZIP_PATH)

WORK_DIR = Path('/content/senatext_modelo') if Path('/content').exists() else Path('senatext_modelo')
VIDEO_ROOT = WORK_DIR / 'Videos'
ARTIFACTS_DIR = WORK_DIR / 'artifacts'
DATASET_PATH = ARTIFACTS_DIR / 'dataset_frames.npz'
LABELS_PATH = ARTIFACTS_DIR / 'labels.json'
CONFIG_PATH = ARTIFACTS_DIR / 'config.json'
MODEL_PATH = ARTIFACTS_DIR / 'senatext_frames_lstm.keras'
EXPORT_ZIP = WORK_DIR / 'senatext_modelo_web.zip'

FRAME_COUNT = 30
FRAME_SIZE = 64
CHANNELS = 1

assert ZIP_PATH.exists(), 'Archivo de videos no cargado.'
WORK_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def slug_label(name):
    text = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_')

if not VIDEO_ROOT.exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(WORK_DIR)

videos = []
for file in sorted(VIDEO_ROOT.rglob('*.mp4')):
    videos.append({
        'path': str(file),
        'label': slug_label(file.parent.name),
        'label_original': file.parent.name,
        'file': file.name,
    })

manifest = pd.DataFrame(videos)
assert not manifest.empty, 'No se encontraron videos MP4 dentro del archivo ZIP.'
resumen_dataset = manifest.groupby(['label_original', 'label']).size().reset_index(name='videos')
display(resumen_dataset)


In [ ]:
def sample_indices(total_frames):
    if total_frames <= 0:
        return np.zeros(FRAME_COUNT, dtype=int)
    return np.linspace(0, total_frames - 1, FRAME_COUNT).astype(int)

def video_to_tensor(video_path, flip=False, brightness=0.0):
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    wanted = set(sample_indices(total_frames))
    frames = []
    current = 0

    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        if current in wanted:
            if flip:
                frame = cv2.flip(frame, 1)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, (FRAME_SIZE, FRAME_SIZE), interpolation=cv2.INTER_AREA)
            frame = np.clip(frame.astype(np.float32) / 255.0 + brightness, 0.0, 1.0)
            frames.append(frame)
        current += 1

    cap.release()

    if not frames:
        frames = [np.zeros((FRAME_SIZE, FRAME_SIZE), dtype=np.float32)]
    while len(frames) < FRAME_COUNT:
        frames.append(frames[-1])
    return np.asarray(frames[:FRAME_COUNT], dtype=np.float32)[..., None]

def build_dataset(rows, augment=False):
    X, labels = [], []
    variants = [(False, 0.0)]
    if augment:
        variants += [(True, 0.0), (False, 0.08), (False, -0.08)]
    for row in rows.itertuples(index=False):
        for flip, brightness in variants:
            X.append(video_to_tensor(row.path, flip=flip, brightness=brightness))
            labels.append(row.label)
    return np.asarray(X, dtype=np.float32), labels


In [ ]:
X_base, labels_base = build_dataset(manifest, augment=False)
encoder = LabelEncoder()
y_base = encoder.fit_transform(labels_base)
class_names = encoder.classes_.tolist()

X_train_raw, X_val, y_train_raw, y_val = train_test_split(
    X_base,
    y_base,
    test_size=0.5,
    random_state=42,
    stratify=y_base,
)

train_rows = manifest.iloc[[i for i in range(len(manifest)) if i not in set(range(len(manifest)))]]
X_aug, labels_aug = build_dataset(manifest, augment=True)
y_aug = encoder.transform(labels_aug)

np.savez_compressed(DATASET_PATH, X=X_aug, y=y_aug, labels=np.asarray(class_names))
LABELS_PATH.write_text(json.dumps(class_names, ensure_ascii=False, indent=2), encoding='utf-8')
CONFIG_PATH.write_text(json.dumps({
    'frame_count': FRAME_COUNT,
    'frame_size': FRAME_SIZE,
    'channels': CHANNELS,
    'input_shape': [1, FRAME_COUNT, FRAME_SIZE, FRAME_SIZE, CHANNELS],
    'normalization': 'grayscale_0_1',
    'classes': class_names,
}, ensure_ascii=False, indent=2), encoding='utf-8')

resumen_preparacion = pd.DataFrame([{
    'videos_originales': len(X_base),
    'muestras_entrenamiento_aumentadas': len(X_aug),
    'clases': len(class_names),
    'frames_por_video': FRAME_COUNT,
    'alto': FRAME_SIZE,
    'ancho': FRAME_SIZE,
    'canales': CHANNELS,
}])
display(resumen_preparacion)


In [ ]:
def create_model():
    model = models.Sequential([
        layers.Input(shape=(FRAME_COUNT, FRAME_SIZE, FRAME_SIZE, CHANNELS)),
        layers.TimeDistributed(layers.Conv2D(16, 3, activation='relu')),
        layers.TimeDistributed(layers.MaxPooling2D()),
        layers.TimeDistributed(layers.Conv2D(32, 3, activation='relu')),
        layers.TimeDistributed(layers.MaxPooling2D()),
        layers.TimeDistributed(layers.Flatten()),
        layers.LSTM(64),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(len(class_names), activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

validation_model = create_model()
history = validation_model.fit(
    X_train_raw,
    y_train_raw,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=4,
    verbose=0,
)
val_loss, val_accuracy = validation_model.evaluate(X_val, y_val, verbose=0)

metricas_validacion = pd.DataFrame([{
    'loss_validacion': float(val_loss),
    'accuracy_validacion': float(val_accuracy),
    'videos_entrenamiento': len(X_train_raw),
    'videos_validacion': len(X_val),
}])
display(metricas_validacion)


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history.history['accuracy'], label='entrenamiento')
plt.plot(history.history['val_accuracy'], label='validacion')
plt.title('Accuracy por epoca')
plt.xlabel('Epoca')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(history.history['loss'], label='entrenamiento')
plt.plot(history.history['val_loss'], label='validacion')
plt.title('Loss por epoca')
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
y_pred = np.argmax(validation_model.predict(X_val, verbose=0), axis=1)
cm = confusion_matrix(y_val, y_pred, labels=np.arange(len(class_names)))

plt.figure(figsize=(12, 10))
plt.imshow(cm, cmap='Blues')
plt.title('Matriz de confusion')
plt.xlabel('Prediccion')
plt.ylabel('Clase real')
plt.xticks(range(len(class_names)), class_names, rotation=90)
plt.yticks(range(len(class_names)), class_names)
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
final_model = create_model()
final_history = final_model.fit(X_aug, y_aug, epochs=100, batch_size=8, verbose=0)
final_loss, final_accuracy = final_model.evaluate(X_aug, y_aug, verbose=0)
final_model.save(MODEL_PATH)

metricas_finales = pd.DataFrame([{
    'loss_entrenamiento_final': float(final_loss),
    'accuracy_entrenamiento_final': float(final_accuracy),
    'muestras_usadas': len(X_aug),
    'modelo': str(MODEL_PATH),
}])
display(metricas_finales)


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(final_history.history['accuracy'], label='accuracy')
plt.plot(final_history.history['loss'], label='loss')
plt.title('Entrenamiento final')
plt.xlabel('Epoca')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
def predict_video(video_path):
    sample = video_to_tensor(video_path)[None, ...]
    probs = final_model.predict(sample, verbose=0)[0]
    order = np.argsort(probs)[::-1][:3]
    return pd.DataFrame({
        'clase': [class_names[i] for i in order],
        'probabilidad': [float(probs[i]) for i in order],
    })

predict_video(manifest.iloc[0]['path'])


In [ ]:
shutil.make_archive(str(EXPORT_ZIP.with_suffix('')), 'zip', ARTIFACTS_DIR)
if Path('/content').exists():
    from google.colab import files
    files.download(EXPORT_ZIP)


In [ ]:
export_manifest = pd.DataFrame([
    {'archivo': 'senatext_frames_lstm.keras', 'uso': 'modelo entrenado para Python/Keras'},
    {'archivo': 'labels.json', 'uso': 'nombres de clases en orden de salida'},
    {'archivo': 'config.json', 'uso': 'parametros de preprocesamiento para replicar en la aplicacion'},
    {'archivo': 'dataset_frames.npz', 'uso': 'dataset procesado para reentrenamiento o auditoria'},
])
display(export_manifest)

shutil.make_archive(str(EXPORT_ZIP.with_suffix('')), 'zip', ARTIFACTS_DIR)
if Path('/content').exists():
    from google.colab import files
    files.download(EXPORT_ZIP)
